In [3]:
from pathlib import Path
import pandas as pd
from narwhals import DataFrame

path_to_csv = Path("../datasets/rnc_dataset_markup.csv")

In [4]:
try:
    df = pd.read_csv(path_to_csv, sep=';', on_bad_lines='skip')
except FileNotFoundError as e:
    print(f"Error: {e}, try to specify path correct.")

In [5]:
df = df.drop(
    columns=["itemId", "split", "targetDetectedMcIds", "targetSplitMcIds", "caseType"]
)

In [6]:
df.tail()



,sourceMcId,sourceMcTitle,description,shouldSplit
2475,101,Ремонт квартир и домов под ключ,Здраствуйте! Облицовка кафелем устройство ГКЛ...,True
2476,101,Ремонт квартир и домов под ключ,Комплексный ремонт квартир \n\nБЕЗ ВЫХОДНЫХ\n\...,False
2477,101,Ремонт квартир и домов под ключ,Ремонт ванных комнат под ключ в Светлогорске. ...,False
2478,101,Ремонт квартир и домов под ключ,"Ремонт квартир, офисов. От штукатурки до плинт...",False
2479,101,Ремонт квартир и домов под ключ,Все види строительство и ремонт\nЦены договорные,False


In [8]:
from utils.description_features_utils import (
    has_bull_markers,
    slash_counter,
    has_slash,
    paragraph_counter,
    word_separately_in_desc,
    count_the_occurrence_of_words_for_separation,
    turkney_count,
)


def features_from_description(df: DataFrame) -> dict:
    desc = df["description"]
    return dict(
        desc_words_count=len(desc.split()),
        desc_length=len(desc),
        has_bull_markers=has_bull_markers(desc),
        has_slashes=has_slash(desc),
        slash_counted=slash_counter(desc),
        paragraph_counted=paragraph_counter(desc),
        turnkey_count=turkney_count(desc),
    )


features_desc = df.apply(features_from_description, axis=1)

df_with_desc_features = pd.concat((df, pd.DataFrame(features_desc.tolist())), axis=1)

In [9]:
from numpy._typing import ArrayLike
##### tokenizer & embeddings block
import torch
from torch import nn
from sentence_transformers import SentenceTransformer

bert_model = SentenceTransformer('cointegrated/rubert-tiny2')

enc = bert_model.encode(sentences=["привет, мир", "я на луне"])

print(type(enc).__name__, len(enc))

def row_to_embeds(df_row, /, bert_model_=bert_model) -> ArrayLike:
    title = df_row["sourceMcTitle"]
    desc = df_row["description"]
    sep = "[SEP]"
    sentence = [f"{title + sep + desc}"]
    return bert_model_.encode(sentences=sentence)





import numpy as np

embeds = [row_to_embeds(df.iloc[i], bert_model) for i in range(len(df))]
mtrx = np.vstack(embeds)
df_with_embeds = pd.DataFrame(data=mtrx).add_prefix("embeds")



/Users/dmitrii/catboost_avito/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 7969.83it/s]
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPE

ndarray 2


In [58]:

from scipy.spatial.distance import pdist
import re
from random import randint

def l2_mean(input_):
    dist = pdist(input_, metric="euclidean")
    return np.mean(dist)

def sentences_to_embeds_with_l2(df_):
    desc = df_["description"]
    text = str(desc).replace("\\r\\n", "\n").replace("\\n", "\n").replace("\r", "\n")
    sentences = [p.strip() for p in re.split(r'(?<=[.!?])\s+|\n+', text) if p.strip()]
    if len(sentences) < 2:
        return 0.0
    sentence_vectors = bert_model.encode(sentences)
    l2mean = l2_mean(input_=sentence_vectors)
    return l2mean

sent_embeddings_with_l2 = [sentences_to_embeds_with_l2(df.iloc[i]) for i in range(len(df))]


In [92]:
df_l2_embeds = pd.DataFrame({"embeds_l2":sent_embeddings_with_l2})


In [37]:


cov_m = np.cov(mtrx, rowvar=False) # covariance matrix
L, M = np.linalg.eigh(cov_m) # computing eigenvectors
print(*sorted(L, reverse=True), sep="\n")

0.04541116866835409
0.022480200479965847
0.012276932057143522
0.010088075381152292
0.007948909744582332
0.005436520898988183
0.005315522949871401
0.004789539677445479
0.004435420639964932
0.004005102751642469
0.0034115428881490556
0.003246503177387544
0.0031906786982448215
0.0029537534817890504
0.0028181755746184314
0.0025535416849112543
0.002461687548436408
0.002364754431106175
0.0022685836761005213
0.0021348480304687564
0.002061184882312692
0.0019959351853802406
0.0018837717885431618
0.0018638942674142327
0.0017606474770289389
0.0016081673505631158
0.0015642821861159772
0.0014968659197215804
0.0014181962974526977
0.0013948680395698097
0.0012664617654386064
0.0012584251967513276
0.0012021198162557143
0.0011676996707652466
0.0011523507832905885
0.001111667977050824
0.0010668965093472951
0.001007449624348607
0.0010027048233268554
0.0009640130260017287
0.0009308483627505675
0.0009231007895779403
0.0008793211402901715
0.0008665421244250174
0.0008286873067964991
0.0008157312883533347
0.000

In [55]:
from random import randint


def vectors_for_title_and_description(df_row, /, bert_model_=bert_model) -> tuple[ArrayLike, ...]:
    desc, title = df_row["description"], df_row["sourceMcTitle"]
    vectors = bert_model_.encode(sentences=[desc, title])
    return vectors[0], vectors[1]

vectors = vectors_for_title_and_description(df.iloc[randint(1, len(df))])

In [58]:
# L2-Norm as feature block
from numpy import linalg

vectors_from_df = [vectors_for_title_and_description(df.iloc[i]) for i in range(len(df))]



In [95]:
df_final = pd.concat([
      df_with_desc_features.reset_index(drop=True),
      df_l2_embeds.reset_index(drop=True),
      df_with_embeds.reset_index(drop=True)
  ], axis=1)

df_final.head()

,sourceMcId,sourceMcTitle,description,shouldSplit,desc_words_count,desc_length,has_bull_markers,has_slashes,slash_counted,paragraph_counted,...,embeds302,embeds303,embeds304,embeds305,embeds306,embeds307,embeds308,embeds309,embeds310,embeds311
0,101,Ремонт квартир и домов под ключ,"Всё виды строительных работ\r\nКачественно, в ...",False,13,85,False,False,0,1,...,0.061496,0.058535,0.019343,0.048706,0.011362,-0.047822,-0.033475,-0.017282,0.071151,-0.023715
1,101,Ремонт квартир и домов под ключ,Профессионально и качественно сделаем ремонт к...,False,107,803,True,False,0,19,...,0.082723,0.043011,0.092585,0.007061,-0.059490,-0.043415,-0.071917,0.053794,0.034265,-0.044075
2,101,Ремонт квартир и домов под ключ,"ремонт квартир, ванной комнате , балкон",False,6,39,False,False,0,0,...,0.080913,0.052698,0.105720,-0.013622,-0.048646,-0.049319,-0.041059,-0.001585,0.036794,-0.034914
3,101,Ремонт квартир и домов под ключ,ЗBОНИТЕ KОHСУЛЬТАЦИЯ БЕCПЛАTНAЯ ПO ТУЛЬСKOЙ ОБ...,True,169,1247,True,True,1,32,...,0.075075,-0.037641,0.001230,0.037845,-0.042515,-0.034750,0.004734,0.057284,0.108178,-0.076892
4,101,Ремонт квартир и домов под ключ,Ремонт квартир любой сложности. Квартиры под к...,False,26,181,False,False,0,0,...,0.072242,0.014644,0.031661,0.029496,-0.020382,-0.030702,0.009245,0.003996,0.094669,-0.075401


In [78]:
print(
    f"Random should split string for repr: \n\n {df_with_desc_features.iloc[1].to_string()}"
)

Random should split string for repr: 

 sourceMcId                                                         101
sourceMcTitle                          Ремонт квартир и домов под ключ
description          Профессионально и качественно сделаем ремонт к...
shouldSplit                                                      False
desc_words_count                                                   107
desc_length                                                        803
has_bull_markers                                                  True
has_slashes                                                      False
slash_counted                                                        0
paragraph_counted                                                   19
turnkey_count                                                        1


In [102]:
from sklearn.model_selection import train_test_split

X = df_final.drop(columns=["shouldSplit", "sourceMcTitle", "description"])
y = df_final["shouldSplit"]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.15, random_state=45, shuffle=True, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_holdout,
    y_holdout,
    test_size=0.5,
    random_state=52,
    shuffle=True,
    stratify=y_holdout,
)


In [111]:
from sklearn.preprocessing import StandardScaler

X_numeric = X_train.select_dtypes(include=[np.number]).fillna(0)
X_array = X_numeric.values.astype('float64')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_numeric)
x_train_mtrx_cov = np.cov(X_scaled, rowvar=False)
L, M = np.linalg.eigh(x_train_mtrx_cov)

print(*sorted(L, reverse=True), sep="\n")


53.40221927283753
28.077397216647867
16.210160245125536
14.819586380676888
10.95471517097209
8.332426652279594
7.6173263162191995
7.146613663914597
6.5918833499577465
6.027776874888011
5.434231312706634
5.06571887382519
4.856775739472505
4.67586032822037
4.209743626764211
3.8884979665412205
3.777304541976035
3.619058175114896
3.5177096584616128
3.264362683465499
3.169368162410378
3.040242127033606
2.8943644739093664
2.8156954809863803
2.7451219857209437
2.537437081140393
2.3205819292447574
2.2657564809372506
2.217100290521485
2.1310865959663197
2.0907853964760106
1.9818568123869844
1.9448240691227652
1.8735549350570586
1.7553344175882386
1.7361456840512501
1.720296308435401
1.67294429219794
1.6016404366089665
1.5225002840339084
1.435789438019721
1.417040458743697
1.4083457798308936
1.336909130844796
1.3100427637800875
1.2835257798485422
1.2462066671591074
1.229938956997265
1.2127570222588335
1.1706385364575693
1.1478884990604972
1.1250437647349698
1.102694508387885
1.0586626646294681
1

In [118]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA



def preprocess_features(df_node):
    return df_node.select_dtypes(include=[np.number]).fillna(0)

X_train_clean = preprocess_features(X_train)
X_val_clean = preprocess_features(X_val)
X_test_clean = preprocess_features(X_test)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)
X_val_scaled = scaler.transform(X_val_clean)
X_test_scaled = scaler.transform(X_test_clean)


pca = PCA(n_components=120, random_state=42)
pca_x_train = pca.fit_transform(X_train_scaled)
pca_x_val = pca.transform(X_val_scaled)
pca_x_test = pca.transform(X_test_scaled)

In [191]:

from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.01,
    l2_leaf_reg=15,
    loss_function="Logloss",
    depth=6,
    eval_metric="AUC",
    auto_class_weights="Balanced",
)

model.fit(
      X=X_train,
      y=y_train,
      eval_set=(X_val, y_val),
      early_stopping_rounds=50,
      use_best_model=True,
      verbose=50,
  )


0:	test: 0.6476821	best: 0.6476821 (0)	total: 74.8ms	remaining: 3m 44s
50:	test: 0.7909177	best: 0.7918638 (33)	total: 1.99s	remaining: 1m 54s
100:	test: 0.8051088	best: 0.8051088 (100)	total: 3.17s	remaining: 1m 30s
150:	test: 0.8092715	best: 0.8107852 (111)	total: 4.21s	remaining: 1m 19s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8107852412
bestIteration = 111

Shrink model to first 112 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=6, eval_metric='AUC', iterations=3000, l2_leaf_reg=15, learning_rate=0.01, loss_function='Logloss')

In [213]:
# 1. Получаем вероятности (два столбца: для 0 и для 1)
y_probs = model.predict_proba(X_test)[:, 1]
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, recall_score

results = []

for threshold in np.arange(0.00, 0.75, 0.02):
    y_pred_c = (y_probs >= threshold).astype(int)
    acc = accuracy_score(y_test, y_pred_c)
    recall = recall_score(y_test, y_pred_c)
    precision = precision_score(y_test, y_pred_c)
    f1_micro = f1_score(y_test, y_pred_c, average="micro")
    f1_macro = f1_score(y_test, y_pred_c, average="macro")
    c_mtrx = confusion_matrix(y_test, y_pred_c)

    best_thr = {
    "threshold_value": threshold,
    "acc": acc,
    "recall": recall,
    "precision": precision,
    "f1_micro": f1_micro,
    "f1_macro": f1_macro,
    }
    results.append(best_thr)

df_res = pd.DataFrame(results)

df_res.to_markdown()


/Users/dmitrii/catboost_avito/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/dmitrii/catboost_avito/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/dmitrii/catboost_avito/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

'|    |   threshold_value |      acc |    recall |   precision |   f1_micro |   f1_macro |\n|---:|------------------:|---------:|----------:|------------:|-----------:|-----------:|\n|  0 |              0    | 0.193548 | 1         |    0.193548 |   0.193548 |   0.162162 |\n|  1 |              0.02 | 0.193548 | 1         |    0.193548 |   0.193548 |   0.162162 |\n|  2 |              0.04 | 0.193548 | 1         |    0.193548 |   0.193548 |   0.162162 |\n|  3 |              0.06 | 0.193548 | 1         |    0.193548 |   0.193548 |   0.162162 |\n|  4 |              0.08 | 0.193548 | 1         |    0.193548 |   0.193548 |   0.162162 |\n|  5 |              0.1  | 0.193548 | 1         |    0.193548 |   0.193548 |   0.162162 |\n|  6 |              0.12 | 0.193548 | 1         |    0.193548 |   0.193548 |   0.162162 |\n|  7 |              0.14 | 0.193548 | 1         |    0.193548 |   0.193548 |   0.162162 |\n|  8 |              0.16 | 0.193548 | 1         |    0.193548 |   0.193548 |   0.162162 |

In [34]:
def predict(source_mc_id: int, title: str, description: str, bert_model):
    row = {
    "sourceMcId": source_mc_id,
    "desc_words_count": len(description.split()),
    "desc_length": len(description),
    "has_bull_markers": has_bull_markers(description),
    "has_slashes": has_slash(description),
    "slash_counted": slash_counter(description),
    "paragraph_counted": paragraph_counter(description),
    "turnkey_count": turkney_count(description),
}
    sep = "[SEP]"
    sentence = [f"{title + sep + description}"]
    embeddings = bert_model.encode(sentences=sentence)
    arr = np.vstack(embeddings)

    pred_df_with_embeds = pd.DataFrame(data=arr).add_prefix("embeds")
    final_to_pred = pd.concat((pd.DataFrame([row]), pred_df_with_embeds), axis=1)
    return model.predict_proba(final_to_pred)





In [18]:
import pandas as pd
from catboost import Pool

description = "Ремонт любой сложности!!! / замена фреона / ремонт компрессора / замена термостата / выезд на дом / гарантия 12 месяцев / чек / запчасти в наличии."
source_mc_title = "Ремонт квартир и домов под ключ"

predict(title=source_mc_title, description=description, bert_model=bert_model, source_mc_id=101)



array([[0.76853747, 0.23146253]])

In [ ]:
print("=== Колонки X_train ===")
print(f"Всего: {len(X_train.columns)}")
print(f"Первые 15: {list(X_train.columns[:15])}")
print(f"Последние 5: {list(X_train.columns[-5:])}")



In [35]:
# Тест 1: Максимальный семантический разрыв + Структурные триггеры
# (Тут и разный смысл, и много слов, и слэши, и слово-триггер "отдельно")
title_1 = "Ремонт холодильников"
description_1 = (
    "Ремонт холодильников любой сложности на дому. / "
    "Отдельно предлагаю услуги по дрессировке собак и выгулу домашних животных. / "
    "Также занимаюсь продажей запчастей для тракторов."
)

# Тест 2: Твой кейс с напольными покрытиями, но "дожатый" по объему
# (Если короткий вариант не сработал, этот должен из-за веса фичи desc_length)
title_2 = "Напольные покрытия"
description_2 = (
    "Укладка ламината, кварцвинила и паркетной доски быстро и качественно. "
    "Также выполняем полный монтаж перегородок из гипсокартона, возведение стен и перепланировку. "
    "Отдельно возможна установка натяжных потолков во всей квартире. Звоните!"
)

# Тест 3: Чистый "Винегрет" (много слэшей и разных ниш)
title_3 = "Услуги сантехника"
description_3 = "Установка крана / Ремонт стиральных машин / Поклейка обоев / Перевозка мебели / Грузчики"

# Функция для запуска тестов
def run_tests():
    for i, (t, d) in enumerate([(title_1, description_1), (title_2, description_2), (title_3, description_3)], 1):
        # source_mc_id берем из твоей категории (например, 101 или 109)
        res = predict(title=t, description=d, bert_model=bert_model, source_mc_id=101)
        print(f"Тест {i}: [No Split, Split] -> {res}")

run_tests()



Тест 1: [No Split, Split] -> [[0.65988039 0.34011961]]
Тест 2: [No Split, Split] -> [[0.42415261 0.57584739]]
Тест 3: [No Split, Split] -> [[0.57071159 0.42928841]]


In [98]:
import matplotlib.pyplot as plt
fi = model.get_feature_importance(prettified=True)
print(fi.head(10))

  Feature Id  Importances
0  embeds221     4.351077
1  embeds304     3.391322
2  embeds183     3.248544
3  embeds260     3.115775
4   embeds42     2.836069
5  embeds230     2.601932
6  embeds192     2.300962
7   embeds28     2.029416
8  embeds234     2.024211
9  embeds298     2.021510
